# 7. Data Validation and Evaluation

모델보다 먼저 prediction 시점, 독립 단위, split, metric을 고정합니다. 수업용 입력에서 누수와 불균형의 실패를 직접 재현합니다.

핵심 셀은 NumPy를 사용합니다. 유료 API, GPU, 외부 데이터 다운로드는 필요하지 않습니다.

## 1. Preprocessing 누수

평균 대입도 data에서 통계를 학습합니다. test의 100이 training fill에 영향을 주면 정보 경계가 샙니다.

In [1]:
import numpy as np
train=np.array([1.,2.,3.])
test=np.array([np.nan,100.])
train_fill=train.mean()
leaked_fill=np.nanmean(np.r_[train,test])
print("Train fill:",train_fill,"Leaked fill:",leaked_fill)
assert train_fill==2 and leaked_fill==26.5
print(np.where(np.isnan(test),train_fill,test))

Train fill: 2.0 Leaked fill: 26.5
[  2. 100.]


## 2. Row Split과 Group Split

새 machine의 성능을 묻는데 같은 ID가 양쪽에 있으면 암기로도 높은 점수가 나올 수 있습니다. 아래 classifier는 ID를 암기하는 일부러 단순한 반례입니다.

In [2]:
machine=np.repeat(np.arange(8),3)
labels=(machine%2).astype(int)
row_train=np.arange(len(machine))%3!=2
row_test=~row_train
def memorize_predict(train_ids,train_y,test_ids):
    memory={int(i):int(y) for i,y in zip(train_ids,train_y)}
    fallback=int(np.mean(train_y)>=.5)
    return np.array([memory.get(int(i),fallback) for i in test_ids])
pred=memorize_predict(machine[row_train],labels[row_train],machine[row_test])
row_accuracy=np.mean(pred==labels[row_test])
print("Row split accuracy:",row_accuracy)
assert row_accuracy==1

Row split accuracy: 1.0


In [3]:
group_train=machine<4
group_test=~group_train
assert set(machine[group_train]).isdisjoint(machine[group_test])
pred=memorize_predict(machine[group_train],labels[group_train],machine[group_test])
group_accuracy=np.mean(pred==labels[group_test])
print("Group split accuracy:",group_accuracy)
assert group_accuracy==.5

Group split accuracy: 0.5


## 3. 불균형 Baseline

19 negative와 1 positive에서 항상 negative로 예측합니다. accuracy만으로 업무 성공을 판정하지 않습니다.

In [4]:
truth=np.array([0]*19+[1])
pred=np.zeros_like(truth)
accuracy=np.mean(truth==pred)
recall=np.sum((truth==1)&(pred==1))/np.sum(truth==1)
print("Accuracy:",accuracy,"Recall:",recall)
assert accuracy==.95 and recall==0

Accuracy: 0.95 Recall: 0.0


## 4. Threshold와 명시한 오류 비용

이 입력은 validation이라고 명시합니다. missed defect 비용 5, extra inspection 비용 1은 수업의 가정입니다.

In [5]:
scores=np.array([.05,.15,.25,.45,.55,.65,.80,.95])
truth=np.array([0,0,1,0,1,0,1,1])
def score_threshold(t):
    p=scores>=t
    tp=int(np.sum(p&(truth==1)));fp=int(np.sum(p&(truth==0)))
    fn=int(np.sum((~p)&(truth==1)));tn=int(np.sum((~p)&(truth==0)))
    return {"TP":tp,"FP":fp,"FN":fn,"TN":tn,"cost":fp+5*fn}
for threshold in [.25,.5,.8]: print(threshold,score_threshold(threshold))
assert score_threshold(.5)=={"TP":3,"FP":1,"FN":1,"TN":3,"cost":6}

0.25 {'TP': 4, 'FP': 2, 'FN': 0, 'TN': 2, 'cost': 2}
0.5 {'TP': 3, 'FP': 1, 'FN': 1, 'TN': 3, 'cost': 6}
0.8 {'TP': 2, 'FP': 0, 'FN': 2, 'TN': 4, 'cost': 10}


## 5. 과제와 해답

- 이 데이터에서 세 threshold 중 validation 비용을 최소화하는 값을 고르세요. 선택한 뒤 test로 다시 고르지 마세요.
- 같은 accuracy .75에서 n=8과 n=80의 불확실성이 왜 다른지 bootstrap으로 비교하세요.

In [6]:
candidates=[.25,.5,.8]
best=min(candidates,key=lambda t:score_threshold(t)["cost"])
assert best==.25
print("Selected on this validation input:",best)
rng=np.random.default_rng(7)
for n in [8,80]:
    correct=np.array([1]*(3*n//4)+[0]*(n//4))
    boot=rng.choice(correct,size=(5000,n),replace=True).mean(axis=1)
    print("n",n,"bootstrap 95% interval",np.quantile(boot,[.025,.975]).round(3))

Selected on this validation input: 0.25
n 8 bootstrap 95% interval [0.375 1.   ]
n 80 bootstrap 95% interval [0.65  0.838]


## 6. 분포 변화와 Metric의 단위

feature 분포 변화는 성능 저하의 직접 증명이 아닙니다. 실제 label이 들어오면 오류도 따로 측정해야 합니다.

In [7]:
old=np.arange(5);new=np.arange(4,9)
print("Feature means:",old.mean(),new.mean())
residual=np.array([-1,0,1,4],dtype=float)
print("MAE",np.mean(abs(residual)),"MSE",np.mean(residual**2),"RMSE",np.sqrt(np.mean(residual**2)))
assert np.mean(abs(residual))==1.5 and np.mean(residual**2)==4.5

Feature means: 2.0 6.0
MAE 1.5 MSE 4.5 RMSE 2.1213203435596424


## 7. Scorecard

입력 version, 예측 시점, group/time 경계, 전처리 학습 범위, baseline, metric, slice count, 확인하지 못한 조건을 기록하세요. 주어진 score의 의미는 이 절차가 정합니다.